### Imports

In [1]:
import warnings

import torch

from src.config import CONFIG
from src.engine import train_kfold, load_and_benchmark
from src.types.dataset_type import DatasetType
from src.types.filterbank_type import FilterbankType
from src.types.model_type import ModelType
from src.utils.hyperparameter_sweep_utils import run_hyperparameter_sweep
from src.utils.plotting_utils import plot_training_history

warnings.filterwarnings('ignore', category=UserWarning)

/Users/jackrong/Developer/bio-inspired-sonar-for-underwater-object-detection/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Constants

In [2]:
MODEL = ModelType.CNN
DATASET = DatasetType.SHIPSEAR
FILTERBANK = FilterbankType.MEL

### Device

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: mps


### Hyperparameter Tuning

In [4]:
if CONFIG.hyperparameter_tuning.should_run:
    run_hyperparameter_sweep(device, MODEL, DATASET, FILTERBANK)

### Training

In [5]:
if CONFIG.should_train:
    results = train_kfold(device, MODEL, DATASET, FILTERBANK)
    for i, fold in enumerate(results['folds']):
        plot_training_history(**fold['epoch_history'])

### Benchmark

In [6]:
benchmark_metrics = load_and_benchmark(device, MODEL, DATASET, FILTERBANK)

Loading 2223 spectrograms...
Finished loading spectrograms. (1s)
Running benchmark on test set...
[Benchmark Results]
 Accuracy: 93.72%
  Macro    — F1: 0.9409 | Precision: 0.9360 | Recall: 0.9486
  Weighted — F1: 0.9382 | Precision: 0.9420 | Recall: 0.9372
  MACs: 59,607,813 | ACs: 0
